# Kaydedilen En ?yi Modeli Y?kleme ve Test Etme

Bu notebook `models/best_model.keras` dosyas?n? y?kler, test accuracy hesaplar, confusion matrix ?izer ve tek bir m?zik dosyas? i?in t?r tahmini yapar.

In [ ]:
from pathlib import Path
import json
import math

import librosa
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split

SEED = 42
GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock",
]


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "models" / "best_model.keras").exists() or (candidate / "dataset" / "features_3.0_sec.json").exists():
            return candidate
    raise FileNotFoundError("Proje k?k? bulunamad?. Notebook'u music-genre-classification i?inde ?al??t?r?n.")

PROJECT_ROOT = find_project_root()
FEATURES_PATH = PROJECT_ROOT / "dataset" / "features_3.0_sec.json"
MODEL_PATH = PROJECT_ROOT / "models" / "best_model.keras"

print(f"Proje k?k?: {PROJECT_ROOT}")
print(f"Model yolu: {MODEL_PATH}")
print(f"Features yolu: {FEATURES_PATH}")

In [ ]:
if not MODEL_PATH.exists():
    fallback = PROJECT_ROOT / "models" / "best_model.h5"
    if fallback.exists():
        MODEL_PATH = fallback
    else:
        raise FileNotFoundError("best_model.keras veya best_model.h5 bulunamad?. ?nce cnn_model.ipynb dosyas?n? ?al??t?r?n.")

model = tf.keras.models.load_model(MODEL_PATH)
model.summary()

In [ ]:
with FEATURES_PATH.open("r", encoding="utf-8") as fp:
    data = json.load(fp)

X = np.array(data["mfcc"], dtype=np.float32)
y = np.array(data["genre_num"], dtype=np.int64)

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_test_cnn = X_test[..., np.newaxis]

test_loss, test_accuracy = model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
y_pred = np.argmax(model.predict(X_test_cnn, verbose=0), axis=1)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 10))
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=GENRES)
display.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix - Best CNN Model")
plt.tight_layout()
plt.show()

## Tek Bir M?zik Dosyas? ??in Tahmin

A?a??daki `SAMPLE_AUDIO_PATH` de?i?kenine istedi?iniz `.wav` dosyas?n?n yolunu yazabilirsiniz. ?rnek olarak veri setinden bir blues par?as? se?ilmi?tir.

In [ ]:
def extract_mfcc_segments_for_prediction(
    file_path: Path,
    sample_rate: int = 22500,
    track_duration: int = 30,
    segment_duration: float = 3.0,
    n_fft: int = 2048,
    hop_length: int = 512,
    n_mfcc: int = 13,
) -> np.ndarray:
    signal, sr = librosa.load(file_path, sr=sample_rate, duration=track_duration)
    samples_per_segment = int(sample_rate * segment_duration)
    expected_frames = math.ceil(samples_per_segment / hop_length)
    num_segments = max(1, math.ceil(len(signal) / samples_per_segment))

    mfcc_segments = []
    for segment_index in range(num_segments):
        start_sample = segment_index * samples_per_segment
        end_sample = start_sample + samples_per_segment
        segment = signal[start_sample:end_sample]

        if len(segment) == 0:
            continue
        if len(segment) < samples_per_segment:
            segment = np.pad(segment, (0, samples_per_segment - len(segment)))

        mfcc = librosa.feature.mfcc(
            y=segment,
            sr=sr,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mfcc=n_mfcc,
        ).T

        if mfcc.shape[0] < expected_frames:
            mfcc = np.pad(mfcc, ((0, expected_frames - mfcc.shape[0]), (0, 0)))
        elif mfcc.shape[0] > expected_frames:
            mfcc = mfcc[:expected_frames, :]

        mfcc_segments.append(mfcc)

    if not mfcc_segments:
        raise ValueError("Tahmin i?in ge?erli ses segmenti ??kar?lamad?.")

    return np.array(mfcc_segments, dtype=np.float32)[..., np.newaxis]


def predict_genre(file_path: Path):
    X_audio = extract_mfcc_segments_for_prediction(file_path)
    probabilities = model.predict(X_audio, verbose=0)
    mean_probabilities = probabilities.mean(axis=0)
    predicted_index = int(np.argmax(mean_probabilities))
    predicted_genre = GENRES[predicted_index]
    return predicted_genre, predicted_index, mean_probabilities

SAMPLE_AUDIO_PATH = PROJECT_ROOT / "dataset" / "genres_original" / "blues" / "blues.00000.wav"
predicted_genre, predicted_index, probabilities = predict_genre(SAMPLE_AUDIO_PATH)

print(f"Dosya: {SAMPLE_AUDIO_PATH}")
print(f"Tahmin: {predicted_genre.upper()} (S?n?f ID: {predicted_index})")
print("Olas?l?klar:")
for genre, probability in zip(GENRES, probabilities):
    print(f"  {genre:10s}: {probability * 100:5.2f}%")